# 03 — Storage: home and scratch

**Home** is working space (run inputs, checkpoints, parameters, own codes, outputs awaiting transfer); a group's
need is attributed to clusters by its workload mix. **Scratch** demand is shown two ways: the *sum of peaks*
(every group's largest concurrent use at once — the worst case) and the *time-averaged* load (jobs per year ×
(wall time + residence after the job) / 8,760 h). Capacities come from each cluster's usable storage and the
committee's home:scratch split.

*Principles: P8, P12 in the [analysis framework](https://github.com/svaradh/hpc-questionnaire/blob/main/docs/analysis-framework.md). Nothing here scores or ranks groups; flags are candidates for the committee to look at. Totals cover responding groups only, and "Don't know" is never counted as zero; a yes/no flag shown as `<NA>` means its inputs are unknown. Only the respondents' answers and the cluster configuration are used — nothing depends on which code a workload runs.*

**To run:** paste the Spreadsheet ID in the first code cell, then *Runtime → Run all*. Colab asks once per session to sign in with your IISER Google account.

In [ ]:
SPREADSHEET_ID = 'PASTE_YOUR_SPREADSHEET_ID_HERE'   # <- the long string in the sheet URL
REPO_REF = 'main'                                   # git branch/tag for the helper code and cluster config

import os, urllib.request
if not os.path.exists('hpc_analysis.py'):            # in Colab: fetch the helper modules once per session
    for f in ['sheets_client.py', 'hpc_analysis.py', 'hpc_plots.py', 'code_capabilities.json']:
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/svaradh/hpc-questionnaire/{REPO_REF}/analysis/{f}', f)

import pandas as pd
from sheets_client import load_data, load_config
import hpc_analysis as ha
import hpc_plots as hp
pd.set_option('display.max_colwidth', 120)

dfs = load_data(SPREADSHEET_ID)
cfg = load_config(ref=REPO_REF)
wl = ha.workloads(dfs, cfg)
cov = ha.coverage(dfs)
print(f"{cov['groups']} groups responded; {cov['workloads']} workloads described "
      f"({cov['groups_with_workloads']} groups with at least one).")

## Home area

In [ ]:
home, home_groups = ha.home_balance(dfs, wl, cfg)
hp.balance(home, 'Home-area working space (TB)', {'demand_now_tb': 'Needed now', 'demand_3y_tb': 'Needed in 3 years'},
           capacity_col='capacity_tb', ylabel='TB')
print(ha.deficit_summary(home, 'demand_now_tb', 'capacity_tb', ' TB', 'now'))
print(ha.deficit_summary(home, 'demand_3y_tb', 'capacity_tb', ' TB', 'in 3 years'))
print(f"Groups with unknown home need: {home.attrs['groups_unknown']}")
home.round(2)

Per group, largest first. `over_guide` uses the committee's home guide per group (not set = `<NA>`);
`outputs_kept_over_3_months` marks groups whose final outputs stay on the cluster for more than 3 months before transfer.

In [ ]:
home_groups.assign(home_now_tb=lambda d: d['home_now_gb'] / 1024).round(2)

## Scratch

In [ ]:
scr, scr_wl = ha.scratch_balance(dfs, wl, cfg)
hp.balance(scr, 'Scratch (TB)', {'peak_sum_tb': 'Sum of peaks (worst case)', 'average_tb': 'Time-averaged'},
           capacity_col='capacity_tb', ylabel='TB')
print(ha.deficit_summary(scr, 'peak_sum_tb', 'capacity_tb', ' TB', 'sum of peaks'))
print(ha.deficit_summary(scr, 'average_tb', 'capacity_tb', ' TB', 'time-averaged'))
scr.round(2)

Per workload, largest peak first. `over_trigger` = peak concurrent scratch above the follow-up trigger;
`over_user_cap` uses the committee's largest-area-per-user (not set = `<NA>`). Both use the peak *with inputs*
(files read + written), the case where only scratch is on the fast parallel file system; the cluster table counts
inputs only for clusters with `storage.scratchOnlyOnFastPFS` set. The Section D "Files and scratch" details (I/O share,
re-reads, local-disk option) tell whether a large need is write-heavy enough to justify scratch.

In [ ]:
display(scr_wl.assign(peak_tb=lambda d: d['scratch_peak_gb'] / 1024).round(2).head(15))
dfs['Scratch']